# 🚆 Train Ridership Forecasting — Model Comparison

This notebook compares the **ElasticNet** and **LSTM** models across four evaluation dimensions:

| # | Method | What it tells you |
|---|--------|-------------------|
| 1 | Metric Comparison | Which model has lower error overall |
| 2 | Visual Prediction Comparison | Where each model over/under-predicts |
| 3 | Per-Station Comparison | Which model works better at each station |
| 4 | Train Allocation Comparison | How differently they allocate trains in practice |

> **Prerequisites:** Both `ElasticNet_Ridership_Forecasting.ipynb` and `LSTM_Ridership_Forecasting.ipynb` must be run first so their saved artifacts (`.pkl`, `.h5`, scalers) are available.

---
## 1. Setup & Imports

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tensorflow.keras.models import load_model

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

# Color palette — consistent across all plots
COLOR_ACTUAL    = '#2c2c2c'
COLOR_ELASTIC   = '#2196F3'   # blue  → ElasticNet
COLOR_LSTM      = '#FF5722'   # orange → LSTM
COLOR_WIN_E     = '#90CAF9'
COLOR_WIN_L     = '#FFAB91'

c:\Users\R.A.B\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.stats import gaussian_kde



[TensorFlow DLL Diagnostic] Analyzing: C:\Users\R.A.B\AppData\Roaming\Python\Python311\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "C:\Users\R.A.B\AppData\Roaming\Python\Python311\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

---
## 2. Rebuild Shared Preprocessing

Both models were trained on the same preprocessed data. We re-run the shared pipeline here to produce a common `test_df` for fair comparison.

In [ ]:
# ── Helper functions (identical to both model notebooks) ──────────────────────

MONTH_MAPPING = {
    'January':1,'February':2,'March':3,'April':4,
    'May':5,'June':6,'July':7,'August':8,
    'September':9,'October':10,'November':11,'December':12
}

def time_split_by_station(df, split_date):
    train_list, test_list = [], []
    for station, g in df.groupby('Station'):
        g = g.sort_values('date')
        train_list.append(g[g['date'] < split_date])
        test_list.append(g[g['date'] >= split_date])
    return pd.concat(train_list), pd.concat(test_list)

def add_calendar_features(df):
    df = df.copy()
    df['month']       = df['date'].dt.month
    df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
    return df

def create_time_series_features(df):
    df = df.copy().sort_values(['Station', 'date'])
    for lag in [1, 2, 3]:
        df[f'lag_{lag}_period'] = df.groupby(['Station','Period'])['Ridership'].shift(lag)
    df['rolling_mean_3_period'] = (
        df.groupby(['Station','Period'])['Ridership']
          .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )
    df['rolling_mean_7_period'] = (
        df.groupby(['Station','Period'])['Ridership']
          .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    )
    return df

def target_encode_station(train_df, test_df, target_col='Ridership'):
    station_mean = train_df.groupby('Station')[target_col].mean()
    global_mean  = train_df[target_col].mean()
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_df['Station_enc'] = train_df['Station'].map(station_mean)
    test_df['Station_enc']  = test_df['Station'].map(station_mean).fillna(global_mean)
    return train_df, test_df

def allocate_trains(df, capacity=600, min_trains=1):
    df = df.copy()
    df['required_trains'] = (
        np.ceil(df['predicted_ridership'] / capacity)
        .astype(int).clip(lower=min_trains)
    )
    return df

In [ ]:
# ── Load & preprocess data ─────────────────────────────────────────────────────

df = pd.read_csv('../data/Ridership.csv')
df['date'] = pd.to_datetime(
    df['Year'].astype(str) + '-' +
    df['Month'].map(MONTH_MAPPING).astype(str) + '-' +
    df['Day'].astype(str)
)

# Filter rare stations (same threshold used in both model notebooks)
MIN_RECORDS    = 30
station_counts = df['Station'].value_counts()
valid_stations = station_counts[station_counts >= MIN_RECORDS].index
df = df[df['Station'].isin(valid_stations)].reset_index(drop=True)
df = df.sort_values('date').reset_index(drop=True)

df = add_calendar_features(df)

SPLIT_DATE = '2022-01-01'
train_df, test_df = time_split_by_station(df, split_date=SPLIT_DATE)

# Outlier removal (train stats only)
Q1 = train_df['Ridership'].quantile(0.25)
Q3 = train_df['Ridership'].quantile(0.75)
IQR = Q3 - Q1
lower_bound  = Q1 - 1.5 * IQR
upper_bound  = Q3 + 1.5 * IQR
median_value = train_df['Ridership'].median()

for split in [train_df, test_df]:
    mask = (split['Ridership'] < lower_bound) | (split['Ridership'] > upper_bound)
    split.loc[mask, 'Ridership'] = median_value

train_df = create_time_series_features(train_df)
test_df  = create_time_series_features(test_df)
train_df = train_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

train_df, test_df = target_encode_station(train_df, test_df)

print(f'Shared test set: {len(test_df):,} rows')
print(f'Stations in test: {sorted(test_df["Station"].unique())}')

---
## 3. Load Saved Models & Generate Predictions

In [ ]:
# ── ElasticNet predictions ─────────────────────────────────────────────────────

elastic_model        = joblib.load('../models/ElasticNet_model/elasticnet_model.pkl')
elastic_preprocessor = joblib.load('../models/ElasticNet_model/preprocessor.pkl')

DROP_COLS   = ['Ridership', 'Station', 'date']
X_test      = test_df.drop(columns=DROP_COLS)
X_test_enc  = elastic_preprocessor.transform(X_test)
y_pred_elastic = elastic_model.predict(X_test_enc)

print(f'ElasticNet predictions shape: {y_pred_elastic.shape}')

In [ ]:
# ── LSTM predictions ───────────────────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.metrics import MeanSquaredError

lstm_model = load_model(
    'lstm_model.h5',
    custom_objects={'mse': MeanSquaredError()}
)
scaler_X      = joblib.load('../models/LSTM_model/scaler_X.pkl')
scaler_y      = joblib.load('../models/LSTM_model/scaler_y.pkl')

NUMERIC_FEATURES = [
    'Station_enc', 'Covid19',
    'month_sin', 'month_cos', 'day_of_week', 'is_weekend',
    'lag_1_period', 'lag_2_period', 'lag_3_period',
    'rolling_mean_3_period', 'rolling_mean_7_period'
]
CATEGORICAL_FEATURES = ['Corridor', 'Workday', 'Period']

# Rebuild the same dummy columns used during LSTM training
cat_train = pd.get_dummies(train_df[CATEGORICAL_FEATURES])
cat_test  = pd.get_dummies(test_df[CATEGORICAL_FEATURES])
cat_test  = cat_test.reindex(columns=cat_train.columns, fill_value=0)

X_test_lstm_raw = pd.concat([test_df[NUMERIC_FEATURES], cat_test], axis=1)
X_test_scaled   = scaler_X.transform(X_test_lstm_raw)
y_test_scaled   = scaler_y.transform(test_df[['Ridership']])

def create_sequences_per_station(X_scaled, y_scaled, station_array, window=7):
    Xs, ys, indices = [], [], []
    for station in np.unique(station_array):
        mask = np.where(station_array == station)[0]
        X_s, y_s = X_scaled[mask], y_scaled[mask]
        for i in range(window, len(X_s)):
            Xs.append(X_s[i-window:i])
            ys.append(y_s[i])
            indices.append(mask[i])
    return np.array(Xs), np.array(ys), np.array(indices)

WINDOW = 7
X_test_seq, y_test_seq, test_indices = create_sequences_per_station(
    X_test_scaled, y_test_scaled, test_df['Station'].values, window=WINDOW
)

y_pred_lstm_scaled = lstm_model.predict(X_test_seq)
y_pred_lstm        = scaler_y.inverse_transform(y_pred_lstm_scaled).flatten()
y_true_lstm        = scaler_y.inverse_transform(y_test_seq).flatten()

# LSTM test_df subset (aligned with sequences)
test_df_lstm = test_df.iloc[test_indices].copy().reset_index(drop=True)

print(f'LSTM predictions shape: {y_pred_lstm.shape}')

In [ ]:
# ── Align both models to the LSTM's test subset (smaller due to windowing) ────
# ElasticNet predictions cover all test rows; LSTM skips the first WINDOW rows
# per station. We align using the tracked indices so both arrays match exactly.

y_pred_elastic_aligned = y_pred_elastic[test_indices]
y_true_aligned         = test_df_lstm['Ridership'].values

print(f'Aligned test size : {len(y_true_aligned):,}')
print(f'ElasticNet preds  : {y_pred_elastic_aligned.shape}')
print(f'LSTM preds        : {y_pred_lstm.shape}')

---
## 4. Method 1 — Metric Comparison

Global performance metrics computed on the same aligned test set.

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    """Compute RMSE, MAE, and R² for a given model."""
    return {
        'Model': model_name,
        'RMSE' : round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        'MAE'  : round(mean_absolute_error(y_true, y_pred), 2),
        'R²'   : round(r2_score(y_true, y_pred), 4)
    }

metrics_elastic = compute_metrics(y_true_aligned, y_pred_elastic_aligned, 'ElasticNet')
metrics_lstm    = compute_metrics(y_true_aligned, y_pred_lstm,            'LSTM')

metrics_df = pd.DataFrame([metrics_elastic, metrics_lstm]).set_index('Model')

# Highlight the winner in each column
def highlight_winner(col):
    if col.name == 'R²':
        winner = col.idxmax()
    else:
        winner = col.idxmin()   # lower RMSE / MAE is better
    return ['font-weight: bold; color: green' if i == winner else '' for i in col.index]

print('\n📊 Global Metrics on Aligned Test Set')
print('=' * 42)
print(metrics_df.to_string())
print('=' * 42)

metrics_df.style.apply(highlight_winner)

In [ ]:
# ── Bar chart comparison ───────────────────────────────────────────────────────

metric_names = ['RMSE', 'MAE', 'R²']
x = np.arange(len(metric_names))
width = 0.35

elastic_vals = [metrics_elastic['RMSE'], metrics_elastic['MAE'], metrics_elastic['R²']]
lstm_vals    = [metrics_lstm['RMSE'],    metrics_lstm['MAE'],    metrics_lstm['R²']]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, (metric, e_val, l_val) in enumerate(zip(metric_names, elastic_vals, lstm_vals)):
    bars = axes[i].bar(
        ['ElasticNet', 'LSTM'],
        [e_val, l_val],
        color=[COLOR_ELASTIC, COLOR_LSTM],
        width=0.5
    )
    # Label on top of each bar
    for bar, val in zip(bars, [e_val, l_val]):
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold'
        )
    axes[i].set_title(metric, fontsize=14)
    axes[i].set_ylabel(metric)
    axes[i].grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle('Method 1 — Global Metric Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Method 2 — Visual Prediction Comparison

Side-by-side plots showing where each model over- or under-predicts.

In [ ]:
# ── 2a: Time series overlay (first 400 samples) ────────────────────────────────

N = 400
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(y_true_aligned[:N],         color=COLOR_ACTUAL,  label='Actual',     linewidth=1.8, zorder=3)
ax.plot(y_pred_elastic_aligned[:N], color=COLOR_ELASTIC, label='ElasticNet', linewidth=1.2, linestyle='--', alpha=0.85)
ax.plot(y_pred_lstm[:N],            color=COLOR_LSTM,    label='LSTM',       linewidth=1.2, linestyle=':',  alpha=0.85)

ax.set_title('Method 2a — Actual vs. Predicted Ridership (first 400 samples)', fontsize=14)
ax.set_xlabel('Sample index')
ax.set_ylabel('Ridership')
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── 2b: Scatter plots ──────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, color, name in zip(
    axes,
    [y_pred_elastic_aligned, y_pred_lstm],
    [COLOR_ELASTIC, COLOR_LSTM],
    ['ElasticNet', 'LSTM']
):
    r2 = r2_score(y_true_aligned, y_pred)
    ax.scatter(y_true_aligned, y_pred, alpha=0.2, s=8, color=color)
    lim = [min(y_true_aligned.min(), y_pred.min()),
           max(y_true_aligned.max(), y_pred.max())]
    ax.plot(lim, lim, 'k--', linewidth=1.2, label='Perfect prediction')
    ax.set_title(f'{name}  (R² = {r2:.4f})', fontsize=13)
    ax.set_xlabel('Actual Ridership')
    ax.set_ylabel('Predicted Ridership')
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.4)

fig.suptitle('Method 2b — Scatter Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2c: Residual distributions ─────────────────────────────────────────────────

residuals_elastic = y_true_aligned - y_pred_elastic_aligned
residuals_lstm    = y_true_aligned - y_pred_lstm

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, residuals, color, name in zip(
    axes,
    [residuals_elastic, residuals_lstm],
    [COLOR_ELASTIC, COLOR_LSTM],
    ['ElasticNet', 'LSTM']
):
    sns.histplot(residuals, bins=60, kde=True, ax=ax, color=color, alpha=0.7)
    ax.axvline(0,                    color='black', linestyle='--', linewidth=1.2, label='Zero error')
    ax.axvline(residuals.mean(),     color='red',   linestyle='-',  linewidth=1.2,
               label=f'Mean = {residuals.mean():.1f}')
    ax.set_title(f'{name} — Residual Distribution', fontsize=13)
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.4)

fig.suptitle('Method 2c — Residual Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Method 3 — Per-Station Comparison

The most important evaluation for this project — comparing model accuracy **at each station individually**, since the end goal is per-station train allocation.

In [ ]:
# ── Compute per-station metrics for both models ────────────────────────────────

test_df_lstm['pred_elastic'] = y_pred_elastic_aligned
test_df_lstm['pred_lstm']    = y_pred_lstm
test_df_lstm['actual']       = y_true_aligned

station_rows = []
for station, grp in test_df_lstm.groupby('Station'):
    e_rmse = np.sqrt(mean_squared_error(grp['actual'], grp['pred_elastic']))
    l_rmse = np.sqrt(mean_squared_error(grp['actual'], grp['pred_lstm']))
    e_r2   = r2_score(grp['actual'], grp['pred_elastic'])
    l_r2   = r2_score(grp['actual'], grp['pred_lstm'])

    station_rows.append({
        'Station'       : station,
        'ElasticNet_RMSE': round(e_rmse, 2),
        'LSTM_RMSE'      : round(l_rmse, 2),
        'ElasticNet_R2'  : round(e_r2, 4),
        'LSTM_R2'        : round(l_r2, 4),
        'RMSE_Winner'    : 'LSTM' if l_rmse < e_rmse else 'ElasticNet',
        'R2_Winner'      : 'LSTM' if l_r2   > e_r2   else 'ElasticNet',
        'RMSE_Delta'     : round(e_rmse - l_rmse, 2),   # positive = LSTM wins
    })

station_df = pd.DataFrame(station_rows).sort_values('RMSE_Delta', ascending=False)

print('Per-Station RMSE Comparison:')
print(station_df[['Station','ElasticNet_RMSE','LSTM_RMSE','RMSE_Winner']].to_string(index=False))

lstm_wins      = (station_df['RMSE_Winner'] == 'LSTM').sum()
elastic_wins   = (station_df['RMSE_Winner'] == 'ElasticNet').sum()
print(f'\nRMSE wins → LSTM: {lstm_wins}  |  ElasticNet: {elastic_wins}')

In [ ]:
# ── Grouped bar chart — RMSE per station ──────────────────────────────────────

stations  = station_df['Station'].values
x         = np.arange(len(stations))
width     = 0.38

fig, ax = plt.subplots(figsize=(max(12, len(stations) * 0.9), 6))

bars_e = ax.bar(x - width/2, station_df['ElasticNet_RMSE'],
                width, label='ElasticNet', color=COLOR_ELASTIC, alpha=0.85)
bars_l = ax.bar(x + width/2, station_df['LSTM_RMSE'],
                width, label='LSTM',       color=COLOR_LSTM,    alpha=0.85)

# Highlight winning bars with a star
for idx, row in station_df.reset_index(drop=True).iterrows():
    winner_bar = bars_l[idx] if row['RMSE_Winner'] == 'LSTM' else bars_e[idx]
    ax.text(
        winner_bar.get_x() + winner_bar.get_width() / 2,
        winner_bar.get_height() + 5,
        '★', ha='center', va='bottom', fontsize=11,
        color='green'
    )

ax.set_title('Method 3 — RMSE per Station  (★ = winner)', fontsize=14)
ax.set_xlabel('Station')
ax.set_ylabel('RMSE')
ax.set_xticks(x)
ax.set_xticklabels(stations, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── Delta chart — how much better is the winner? ──────────────────────────────
# Positive = LSTM wins by that margin; Negative = ElasticNet wins

colors = [COLOR_LSTM if d > 0 else COLOR_ELASTIC for d in station_df['RMSE_Delta']]

fig, ax = plt.subplots(figsize=(max(12, len(stations) * 0.9), 5))
ax.bar(station_df['Station'], station_df['RMSE_Delta'], color=colors)
ax.axhline(0, color='black', linewidth=1)

# Legend patches
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLOR_LSTM,    label='LSTM wins (positive)'),
    Patch(facecolor=COLOR_ELASTIC, label='ElasticNet wins (negative)')
]
ax.legend(handles=legend_elements, fontsize=11)
ax.set_title('RMSE Delta per Station  (ElasticNet RMSE − LSTM RMSE)', fontsize=13)
ax.set_xlabel('Station')
ax.set_ylabel('RMSE Improvement (ElasticNet − LSTM)')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

---
## 7. Method 4 — Train Allocation Comparison

The ultimate business check: do the two models allocate **a different number of trains** in practice?
A model with good RMSE but erratic train allocation is less useful operationally.

In [ ]:
TRAIN_CAPACITY = 600

# Build allocation DataFrames for both models
alloc_elastic = test_df_lstm.copy()
alloc_elastic['predicted_ridership'] = y_pred_elastic_aligned
alloc_elastic = allocate_trains(alloc_elastic, capacity=TRAIN_CAPACITY)
alloc_elastic['model'] = 'ElasticNet'

alloc_lstm = test_df_lstm.copy()
alloc_lstm['predicted_ridership'] = y_pred_lstm
alloc_lstm = allocate_trains(alloc_lstm, capacity=TRAIN_CAPACITY)
alloc_lstm['model'] = 'LSTM'

# Ground truth allocation
alloc_actual = test_df_lstm.copy()
alloc_actual['predicted_ridership'] = y_true_aligned
alloc_actual = allocate_trains(alloc_actual, capacity=TRAIN_CAPACITY)
alloc_actual['model'] = 'Actual'

combined = pd.concat([alloc_elastic, alloc_lstm, alloc_actual])

In [ ]:
# ── 4a: Global allocation distribution ────────────────────────────────────────

print('Required Trains — Descriptive Statistics:')
print(
    combined.groupby('model')['required_trains']
    .describe()
    .round(2)
    .to_string()
)

In [ ]:
# ── 4b: Boxplot of required trains ────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))

order  = ['Actual', 'ElasticNet', 'LSTM']
colors = ['#90A4AE', COLOR_ELASTIC, COLOR_LSTM]

sns.boxplot(
    data=combined, x='model', y='required_trains',
    order=order, palette=dict(zip(order, colors)), ax=ax
)

ax.set_title('Method 4a — Distribution of Required Trains', fontsize=13)
ax.set_xlabel('Model')
ax.set_ylabel('Required Trains')
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4c: Average trains per station ────────────────────────────────────────────

avg_trains = (
    combined
    .groupby(['model', 'Station'])['required_trains']
    .mean()
    .reset_index()
)

pivot_trains = avg_trains.pivot(index='Station', columns='model', values='required_trains')

fig, ax = plt.subplots(figsize=(max(12, len(stations) * 0.9), 5))

x     = np.arange(len(pivot_trains))
width = 0.28

ax.bar(x - width,  pivot_trains.get('Actual',     0), width, label='Actual',     color='#90A4AE', alpha=0.9)
ax.bar(x,          pivot_trains.get('ElasticNet', 0), width, label='ElasticNet', color=COLOR_ELASTIC, alpha=0.85)
ax.bar(x + width,  pivot_trains.get('LSTM',       0), width, label='LSTM',       color=COLOR_LSTM,    alpha=0.85)

ax.set_title('Method 4b — Average Required Trains per Station', fontsize=13)
ax.set_xlabel('Station')
ax.set_ylabel('Avg Required Trains')
ax.set_xticks(x)
ax.set_xticklabels(pivot_trains.index, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4d: Allocation error — how far off from actual trains needed? ──────────────
# Over-allocation = waste; Under-allocation = passengers left behind

alloc_elastic['train_error'] = alloc_elastic['required_trains'] - alloc_actual['required_trains']
alloc_lstm['train_error']    = alloc_lstm['required_trains']    - alloc_actual['required_trains']

print('Allocation Error (Predicted Trains − Actual Trains):')
print()
for model_name, alloc_df in [('ElasticNet', alloc_elastic), ('LSTM', alloc_lstm)]:
    err = alloc_df['train_error']
    over   = (err > 0).sum()
    under  = (err < 0).sum()
    exact  = (err == 0).sum()
    print(f'{model_name}:')
    print(f'  Over-allocation  (waste)     : {over:,}  ({100*over/len(err):.1f}%)')
    print(f'  Under-allocation (shortage)  : {under:,}  ({100*under/len(err):.1f}%)')
    print(f'  Exact match                  : {exact:,}  ({100*exact/len(err):.1f}%)')
    print(f'  Mean absolute error (trains) : {err.abs().mean():.3f}')
    print()

In [ ]:
# ── 4e: Heatmap — average trains per Station × Period for each model ───────────

PERIOD_ORDER = ['AM Peak', 'Midday', 'PM Peak', 'Evening', 'Weekend/Holiday']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, alloc, title in zip(
    axes,
    [alloc_actual, alloc_elastic, alloc_lstm],
    ['Actual', 'ElasticNet', 'LSTM']
):
    pivot = alloc.pivot_table(
        index='Station', columns='Period',
        values='required_trains', aggfunc='mean'
    ).reindex(columns=PERIOD_ORDER)

    sns.heatmap(
        pivot, annot=True, fmt='.1f',
        cmap='YlOrRd', linewidths=0.4,
        ax=ax, vmin=1, vmax=pivot.values.max(),
        cbar=False
    )
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Period')
    ax.set_ylabel('Station' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('Method 4c — Avg Required Trains: Station × Period', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Final Summary

A consolidated view of all four comparison methods.

In [ ]:
lstm_rmse_wins    = (station_df['RMSE_Winner'] == 'LSTM').sum()
elastic_rmse_wins = (station_df['RMSE_Winner'] == 'ElasticNet').sum()
total_stations    = len(station_df)

e_train_mae = alloc_elastic['train_error'].abs().mean()
l_train_mae = alloc_lstm['train_error'].abs().mean()

summary = pd.DataFrame([
    {
        'Criterion'  : 'Global RMSE',
        'ElasticNet' : metrics_elastic['RMSE'],
        'LSTM'       : metrics_lstm['RMSE'],
        'Winner'     : 'LSTM' if metrics_lstm['RMSE'] < metrics_elastic['RMSE'] else 'ElasticNet'
    },
    {
        'Criterion'  : 'Global MAE',
        'ElasticNet' : metrics_elastic['MAE'],
        'LSTM'       : metrics_lstm['MAE'],
        'Winner'     : 'LSTM' if metrics_lstm['MAE'] < metrics_elastic['MAE'] else 'ElasticNet'
    },
    {
        'Criterion'  : 'Global R²',
        'ElasticNet' : metrics_elastic['R²'],
        'LSTM'       : metrics_lstm['R²'],
        'Winner'     : 'LSTM' if metrics_lstm['R²'] > metrics_elastic['R²'] else 'ElasticNet'
    },
    {
        'Criterion'  : 'Per-Station RMSE wins',
        'ElasticNet' : f'{elastic_rmse_wins}/{total_stations}',
        'LSTM'       : f'{lstm_rmse_wins}/{total_stations}',
        'Winner'     : 'LSTM' if lstm_rmse_wins > elastic_rmse_wins else 'ElasticNet'
    },
    {
        'Criterion'  : 'Train Allocation MAE',
        'ElasticNet' : round(e_train_mae, 3),
        'LSTM'       : round(l_train_mae, 3),
        'Winner'     : 'LSTM' if l_train_mae < e_train_mae else 'ElasticNet'
    },
])

print('\n' + '=' * 58)
print('  📋  FINAL COMPARISON SUMMARY')
print('=' * 58)
print(summary.to_string(index=False))
print('=' * 58)

overall_winner_counts = summary['Winner'].value_counts()
overall_winner = overall_winner_counts.idxmax()
print(f'\n🏆  Overall recommended model: {overall_winner}')
print(f'   (wins {overall_winner_counts[overall_winner]} out of {len(summary)} criteria)')